# SummarizationMiidleware中间件
## 作用：对历史消息列表进行摘要&总结，达到压缩上下文的效果

### 原理：在达到触发条件时，调用大模型对历史消息进行摘要，将摘要的结果作为HumanMessage，放到消息列表最开始的位置。

#### 参数说明
 1.model-用于摘要的模型
    可以是模型名称也可以是模型对象，如果传递的是模型名称，底层回调用init_chat_model初始化模型。
 2. trigger- 摘要触发条件
        是以列表，每一个元素对应一个条件，当任一条件满足时，触发摘要。
         1.tokens：token的数量，历史token的累计数量达到该值触发阈值
          2. messages：消息历史数量，历史消息条数达到该值触发摘要
          3. fraction：上下文长度比例，历史token的累计数量达到模型的max_input_tokens*fraction触发摘要
           注意：如果条件包含fraction，那么要求模型的profile中必须包含max_input_tokens，Deepseek模型的profile为空，此时需要手动条件该配置项。
 3.keep- 摘要时保留的原始数据
        支持三种条件，但和trigger不同，keep同一时间只接受一种条件
         同上三个参数一样
 4.token_counter- 统计token数量的函数
    默认使用Langchain提供的count_tokens_approximately,一般不用修改。
    底层逻辑：对应纯文本消息，该函数的大致思路是先统计消息的字符数，也就是len(字符串)，然后再除以每个token大致的字符数，转换为粗略的token数，再加一些额外开销，作为估算的token数。
 5.summary_prompt-摘要时的自定义提示词
    该提示词需要包含{messages}占位符，使得历史消息列表可以被插入。不指定则使用内置提示词。
 6.trim_token_to_summarize-摘要时历史消息的最大token数
    如果历史消息token数大于该值，则会被裁剪，默认为"4000"。
    如果trigger用token作为度量，调大触发阈值时，当前配置项应相应调整，否则会丢失信息。


举例1：

In [9]:
import os
import re
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from openai import base_url, api_key
from pywin.framework.toolmenu import tools

#从.env中加载环境变量
load_dotenv(override=True)

## 自定义模型配置,Deepseek的profile中没有这个，所以一定要加，不加就会报错
custom_profile = {
    "max_input_tokens": 128_000
}

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    model_name="deepseek",
    profile=custom_profile,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)


In [10]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

messages = [
    SystemMessage("你是个非常友好的Ai助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好啊老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴的太早了"),
    HumanMessage("好的，我明白了"),
]

agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.01)
            ],
            keep=("messages", 2),
        )
    ],
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
老王 (the user) introduced himself and asked who the AI is. The initial exchange is a simple greeting/introduction — no concrete task or deliverable has been requested yet.

## SUMMARY
- The user introduced himself as 老王 ("Lao Wang").
- The AI responded introducing itself as 小王 ("Xiao Wang"), in a friendly tone (per system instruction: 你是个非常友好的Ai助手 — the assistant should be very friendly).
- The user acknowledged the introduction: "好的小王，很高兴认识你" ("Okay Xiao Wang, nice to meet you").
- No decisions, strategies, or rejected options were discussed.
- Relevant detail: the agreed in-conversation naming convention is user = 老王, assistant = 小王.

## ARTIFACTS
None.

## NEXT STEPS
- Wait for 老王 to specify his actual task or question; no pending work has been requested yet.
- Maintain the friendly persona and the 小王 self-reference when continuing.
=====

举例2
测试summary_prompt参数

In [11]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

#从.env中加载环境变量
load_dotenv(override=True)

## 自定义模型配置,Deepseek的profile中没有这个，所以一定要加，不加就会报错
custom_profile = {
    "max_input_tokens": 128_000
}

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    model_name="deepseek",
    profile=custom_profile,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

messages = [
    SystemMessage("你是个非常友好的Ai助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好啊老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴的太早了"),
    HumanMessage("好的，我明白了"),
]

agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.01)
            ],
            keep=("messages", 2),
            summary_prompt="请将历史消息进行摘要，消息列表如下\n{messages}"
        )
    ],
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要：

- 系统设定：AI助手应非常友好。
- 老王向AI打招呼并自我介绍，询问AI是谁。
- AI自称“小王”并回应问候。
- 老王表示很高兴认识“小王”。
================================== Ai Message ==================================

你高兴的太早了
================================ Human Message =================================

好的，我明白了
================================== Ai Message ==================================

哈哈，那我先把“高兴”收一收，等真帮上忙再高兴～老王，接下来有什么需要我做的，随时叫我。


# 2.HumanInTheLoopMiddleware中间件
HumanInTheLoopMiddleware(人在环中间件，人工审核中间件)在工具调用前中断Agent运行，等待用户对工具调用请求决策。可选的决策有：approve（同意执行）、edit（编辑调用配置后执行）、reject（拒绝执行）
### 参数
    1.inerrupt_on- 工具名和中断策略的映射
    策略可以是True、False或者InterruptOnConfig对象，精确控制决策选项。
    比如:
    interrupt_on = {
        "get_weather": True,
        "read_email_toold": False,
        "send_email_tool":{
            "allowed_description":["approve","edit"],
        }
}
    这里面的interruptOnConfig是一个TypedDict的子类，里面有allowed_description和description两个参数，description的优先级高于description_predix，后者回更改所有工具中断的描述
    2.description_prefix-自定义终端描述
    默认为"Tool execution requires approval"



举例1：调用前中断

In [22]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)
from langchain_core.tools import tool

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    model_name="deepseek",
    profile=custom_profile,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
    city (str): 城市名称
    is_forcast (bool): 是否包含明天天气预报
    """
    res = f"{city} 今天天气不错"
    if is_forcast:
        res += "\n明天下雨"


@tool
def get_news() -> str:
    """
    查询当日新闻
    :return:
    """
    return "委内瑞拉被美国闪电击破"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""

    return f"邮件ID: {email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发生邮件伪函数"""
    print(">>>真的执行发生邮件工具了")
    return f"发送给 {recipient} 的邮件标题是： {subject},内容是: {body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦！"
                }
            },
            description_prefix="中断喽~"
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

# 第一次调用，会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气"
                                 "查询今日新闻"
                                 "查看ID为 'sk2131421' 的邮件内容，"
                                 "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                 "同时做这四件事")
        ]
    },
    config=config
)
print("======第一次 invoke 返回======")
print("======原始响应======")
rprint(response)
print("======= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

#关键：查看中断信息
interrupts = response.get("__interrupt__", [])
print("=========== interrputs ===========")
rprint(interrupts)
print("====逐个打印 interrupt 请求=======")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)

======第一次 invoke 返回======
======原始响应======


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='2e9c216a-dc31-41fc-a198-06a13d6a6aeb'
        ),
        AIMessage(
            content='我会同时帮你完成这四件事：',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user wants four things done at once. All are independent, so I should 
call all four tools in the same block.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 206,
                    'prompt_tokens': 536,
                    'total_tokens': 742,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 25,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 152
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': 'cf497766-5b33-4ac9-9d9f-aee68edf440d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14d38-21af-7b60-ac28-6fb204af2268-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_00_nZYPgpld2xgCks6lfoGf5975',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_r3HgwauHeolblQCqUBAs7644', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_fKfvfN9oyi4Vpj8huZKl3874',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_nW2jFTsl5MfoXOW6Pdun7999',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 536,
                'output_tokens': 206,
                'total_tokens': 742,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {'reasoning': 25}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断喽~\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断喽~\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断啦！'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
     

======= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我会同时帮你完成这四件事：
Tool Calls:
  get_weather (call_00_nZYPgpld2xgCks6lfoGf5975)
 Call ID: call_00_nZYPgpld2xgCks6lfoGf5975
  Args:
    city: 北京
    is_forcast: False
  get_news (call_01_r3HgwauHeolblQCqUBAs7644)
 Call ID: call_01_r3HgwauHeolblQCqUBAs7644
  Args:
  read_email_tool (call_02_fKfvfN9oyi4Vpj8huZKl3874)
 Call ID: call_02_fKfvfN9oyi4Vpj8huZKl3874
  Args:
    email_id: sk2131421
  send_email_tool (call_03_nW2jFTsl5MfoXOW6Pdun7999)
 Call ID: call_03_nW2jFTsl5MfoXOW6Pdun7999
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
=========== interrputs ===========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'description': "中断喽~\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断喽~\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断啦！'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='bed874ca6dd55079d75729643f0554c7'
    )
]

====逐个打印 interrupt 请求=======


{
    'name': 'get_weather',
    'args': {'city': '北京', 'is_forcast': False},
    'description': "中断喽~\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
}

{'name': 'get_news', 'args': {}, 'description': '中断喽~\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
    'description': '发送邮件中断啦！'
}

In [23]:
from langgraph.types import Command

# 如果有中断，说明进入人在环了
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "中国上海市", "is_forcast": True}
    }
}

new_decision = {
    "type": "approve",
}

send_email_decision = {
    "type": "approve"
}
decisions = {
    "decisions": []
}

#决策的顺序必须和返回的中断请求顺序一致
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(new_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    # 审核通过，继续执行
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config,  #必须是同一个thread_id
    )

print("======= 审批后继续执行 ========")
for msg in resumed_response["messages"]:
    msg.pretty_print()

>>>真的执行发生邮件工具了
======= 审批后继续执行 ========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我会同时帮你完成这四件事：
Tool Calls:
  get_weather (call_00_nZYPgpld2xgCks6lfoGf5975)
 Call ID: call_00_nZYPgpld2xgCks6lfoGf5975
  Args:
    city: 中国上海市
    is_forcast: True
  get_news (call_01_r3HgwauHeolblQCqUBAs7644)
 Call ID: call_01_r3HgwauHeolblQCqUBAs7644
  Args:
  read_email_tool (call_02_fKfvfN9oyi4Vpj8huZKl3874)
 Call ID: call_02_fKfvfN9oyi4Vpj8huZKl3874
  Args:
    email_id: sk2131421
  send_email_tool (call_03_nW2jFTsl5MfoXOW6Pdun7999)
 Call ID: call_03_nW2jFTsl5MfoXOW6Pdun7999
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

null
================================= Tool Mes

# 3.PIIMiddleware中间件
敏感信息保护
PII中间件用于检测和处理对话中的个人身份信息，支持自定义处理策略
### 参数
1. pii_types- 检测的PII类型
    - "all": 检测所有PII类型
    - "name": 检测个人姓名
    - "ssn": 检测社会保障号
    - "credit_card": 检测信用卡号
    - "phone_number": 检测电话号码
    - "email": 检测邮箱地址
    - "url": 网址
2. strategy- 处理PII信息的策略
    支持四种选择
    - "redact": 将检测到的PII信息用字符串[REDACTED_[PII_TYPE]]替换，其中的PII_TYPE是上面提到的具体类型，比如[REDACTED_EMAIL]、[READCTED_CREDIT_CARD]这样的标签。完全"擦除/隐藏"真是内容。适用日志清洗、合规需求、公开输出时隐藏敏感信息。
    - "mask": 用***代替PII信息，比如信用卡号可能变成****_****_****_1234(只保留最后几位/部分可见)，邮箱可能保留域名部分+隐藏用户名的一部分等-既隐藏大部分敏感信息，又保留了一点"可辨识性"(比如账号的后四位、域名等),适用用户服务界面/前端显示/需要部分可识别但不泄露完整敏感内容的场景
    - "hash": 用检测到的PII信息的哈希值替代原值。比如<email_hash:a1b2c3d4>。使用analytics、debug、统计分析、匿名追踪等场景。
    - "block": 如果检测到PII信息，直接抛出异常。适用对隐私要求极高、绝不允许泄露任何敏感信息的场景。

3. 参数3：detector-自定义PII检测函数或者正则表达式
    如果没有提供则使用内置的检测函数。
    langchain为每种PII信息定制了专门的检测函数，相关源码如下
    BUILTIN_DETECTORS: dict[str, Detector] = {
        "email": detect_email,
        "credit_card": detect_credit_card,
        "ip": detect_ip,
        "mac_address": detect_mac_address,
        "url": detect_url,
        }
4. 参数4：apply_to_input-是否在调用模型前检测
5. 参数5.apply_to_output-是否在模型调用后检测
6. apply_to_tool_results-是否在工具调用后检测其输出
    默认为False。通常我们只在模型调用前检测。因为PII检测的主要目的就是避免将敏感信息发送给模型服务导致信息泄露。



##### 举例1：使用内置检测器

In [9]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from rich import print as rprint
from langchain_core.messages.human import HumanMessage
from langchain.agents.middleware import PIIMiddleware

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

try:
    response1 = agent.invoke({
        "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

我无法代为执行以下操作：发送邮件、查询银行卡余额、访问未知链接，以及确认真实 MAC 地址；这些涉及账号权限、隐私和外部访问安全。建议你通过银行官方 App/网银/客服查询余额，并不要在其他渠道提供完整银行卡号。

关于 MAC 地址：  
`**-**-**-**-**-11` **不是完整的有效 MAC 地址**。MAC 地址通常应为 6 组两位十六进制字符，例如：`00-1A-2B-3C-4D-11`。你给的形式像是“被掩码的 MAC 地址”，但 `*` 不是有效十六进制字符，因此无法据此确认真实 MAC 地址或厂商。

我可以帮你：
1. 起草要发送的邮件内容，由你自行发送；  
2. 解释如何安全查询银行卡余额；  
3. 在你提供完整、非敏感的 MAC 地址后校验其格式；  
4. 帮你判断链接是否可疑（请勿直接访问未知链接）。
============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content


#### 举例2：自定义检测器/函数

In [12]:
import re


# 自定义检测函数
def detect_phont_number(content: str):
    return [
        {
            "text": m.group(0),  # 提取出具体匹配到的 11 位数文本 （例如"13812345678"）
            "start": m.start(),  # 这段数字在原文中的"起始索引位置"(从0开始算)
            "end": m.end()  # 这段数字在原文本中的"结束索引位置"
        } for m in re.finditer(r"[0-9]{11}", content)
    ]


# 测试
text = "我的电话号码为:15912345678，我爸的电话号码为: 17770012345。"
result = detect_phont_number(text)
print(result)

[{'text': '15912345678', 'start': 8, 'end': 19}, {'text': '17770012345', 'start': 30, 'end': 41}]


In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from rich import print as rprint
from langchain_core.messages.human import HumanMessage
from langchain.agents.middleware import PIIMiddleware

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phont_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
        这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
        帮我给这个号码打电话： 12345612345
        访问 https://localhost:1234
    """)]
})
for msg in response["messages"]:
    msg.pretty_print()


**re.finditer(pattern, content)** 是 Python 正则模块中非常高效的一个方法，它会在 **content** 字
符串中从左到右扫描，每当找到一个符合条件的 11 位数字，它不会立刻把字符串提取出来，而是
生成一个 **匹配对象**。它返回的是一个迭代器（Iterator）。

# TodoListMiddleware中间件
TodoListMiddleware中间件赋予了Agent **任务规划** 和 **追踪进度** 的能力，可以 **应对复杂的多步任务** 。
比如，当一个大任务需要被拆解为 **3 个以上的子任务**，且前面的步骤是后面步骤的前提时，如果不列
Todo 列表，大模型在执行到第 3 步时，很容易**忘记自己最初的目标**，或者在工具返回大量报错信息后
“应激”，直接跳过验证去回答用户。
此时， **TodoListMiddleware** 中间件强制它把计划挂在全局状态里，时刻提醒它“下一步该干什么”。

你的任务是否需要拆解？
    ├── 否 (比如：问答、翻译、单次函数调用) ──> ❌ 绝不需要，浪费算力
    └── 是 (比如：写一个包含多文件的工程)
    └── 步骤是否多变且需要应对失败？
        ├── 否 (步骤完全固定，如 A->B->C) ──> ❌ 传统的LangGraph线性节点即可
        └── 是 (AI 需要边做边调计划) ──> 引入 TodoListMiddleware
如果把普通的 Agent 比作“想到哪写到哪”的实习生，那么引入了 TodoListMiddleware 的 Agent
就是“先写方案、再列 CheckList、最后按部就班执行”的资深工程师。

典型场景：
任务链路长、步骤多，且有严格的先后依赖关系
需要在前端 UI 界面实时展示 Agent 的“思考与执行进度”
....
To-do list的创建和维护是通过调用 write_todos工具 实现的。
## 参数说明
    1. system_prompt —自定义指导todo列表使用的提示词
        不提供则使用内置提示词，通常不必提供。
    2. tool_description —自定义write_tools工具的描述信息
        不提供则使用内置描述，通常不必提供。

模型初始化

In [14]:
load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_API_BASE")
)


In [16]:
# 提供工具列表
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("../todo-wordspace")


@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path：工作区下的相对路径，一定指向目录，默认为.,表示工作区根路径，不能访问工作区外的目录
    :param path:
    :return:
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在: {path}"

    if not target.is_dir():
        return f"错误，不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))

    if not items:
        return f"目录为空: {path}"

    lines = []

    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)


@tool
def read_file(path: str) -> str:
    """
    读取工作区的文本文件内容。 path 只能是相对路径

    Args:
        path: 工作区内的文件
    :param path:
    :return:
    """

    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误： 只允许读取工作区内的文件"

    if not file_path.exists():
        return f"错误：文件不存在: {path}"

    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的文本内容
    :param path:
    :param content:
    :return:
    """

    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误，只允许写入工作区内的文件。"

    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q,并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    :return:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败：{e}"

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage
from rich import print as rprint
#1. 初始化 Agent
agent = create_agent(
    model=model,
    # write_todos等工具，TodolistMiddleware需要配合这些工具使用
    tools=[list_files, read_file, write_file, run_tests],
    # 引入 todo 列表中间件
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手。遇到多步骤任务时，显示有 write_todos 指定代办事项;"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下运行。"
    ),
)

# 2.使用invoke进行同步调用
print("正在执行 Agent 任务")

final_state = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
        ]
    }
)
rprint(final_state)

正在执行 Agent 任务


{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下 my_add.py 文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='f60b3285-1df2-4cd5-b49a-54250aba15b4'
        ),
        AIMessage(
            content='我先查看工作区的结构和相关文件。',
            additional_kwargs={'refusal': None, 'reasoning_content': ''},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 74,
                    'prompt_tokens': 1786,
                    'total_tokens': 1860,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 1786
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '28bc2387-d364-4101-9f6a-8d5c6e16184f',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14e46-23a1-7083-9d95-824d03047903-0',
            tool_calls=[
                {
                    'name': 'list_files',
                    'args': {'path': '.'},
                    'id': 'call_00_ET_wqRR5AJoSiZAyl7srhEa7995',
                    'type': 'tool_call'
                },
                {
                    'name': 'read_file',
                    'args': {'path': 'my_add.py'},
                    'id': 'call_01_ET_Xf9VUamXMq1HaBAmDH3H7305',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1786,
                'output_tokens': 74,
                'total_tokens': 1860,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='[DIR] .pytest_cache\n[DIR] __pycache__\n[FILE] my_add.py\n[FILE] test_my_add.py',
            name='list_files',
            id='644b9d37-bbb4-4996-8374-9f1ca134f48b',
            tool_call_id='call_00_ET_wqRR5AJoSiZAyl7srhEa7995'
        ),
        ToolMessage(
            content='def add(a: int, b: int) -> int:\n    """返回两个整数的和"""\n    return a - b\n',
            name='read_file',
            id='f4fff9d6-9790-4b2a-a959-65bf2ce134c7',
            tool_call_id='call_01_ET_Xf9VUamXMq1HaBAmDH3H7305'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None, 'reasoning_content': "Let's read test file and run tests."},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 64,
                    'prompt_tokens': 1936,
                    'total_tokens': 2000,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 9,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1792},
                    'prompt_cache_hit_tokens': 1792,
                    'prompt_cache_miss_tokens': 144
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '0a0e739d-ab4b-46ed-8e2b-aabfefa8f122',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
         